In [58]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [60]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [63]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "'''json")
    text = chat(messages, stop_sequences=["'''"])
    return json.loads(text)

In [64]:
dataset = generate_dataset()
dataset

[{'task': 'Write a regular expression to validate AWS S3 bucket names. S3 bucket names must be 3-63 characters long, contain only lowercase letters, numbers, and hyphens, start and end with a letter or number, and cannot contain consecutive hyphens.'},
 {'task': 'Write a Python function that takes an AWS CloudFormation template (as a dictionary) and returns a list of all the resource logical IDs that have a DependsOn property.'},
 {'task': "Write a JSON object representing an AWS IAM policy that grants read-only access to a specific S3 bucket named 'my-data-bucket' and all its objects."}]

In [91]:
def run_prompt(test_case):
    """Merges the promt and test case input, then returns the results"""
    prompt = f"""
Please solve the following task:
{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [92]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    #TODO
    score = 10
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }
    output = run_prompt(test_case)
    # Check if the output is correct. This will depend on the specific task and expected output format.
    # For example, if the expected output is a JSON object, we can try to parse the output as JSON and compare it to the expected output.

In [93]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each test case"""
    results = []
    for test_case in dataset:
        print(test_case)
        result = run_test_case(test_case)
        results.append(result)
    return results

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)